# UniArchive — Python quickstart

`uniarchive` is a Cython extension over the UniArchive C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-uniarchive
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## Creating an archive

`create` takes an output path and the paths to archive. It never replaces an
existing output — a caller that wants replacement removes the file first.

In [1]:
import os, tempfile, uniarchive

work = tempfile.mkdtemp()
os.chdir(work)
os.makedirs("data", exist_ok=True)
# newline="": text mode would write CRLF on Windows, changing both the bytes
# read back and the archive's size.
open("notes.txt", "w", newline="").write("one\n")
open("data/values.csv", "w", newline="").write("a,b\n1,2\n")

uniarchive.create("demo.zip", ["notes.txt", "data"])
uniarchive.version(), os.path.getsize("demo.zip")

('0.1.0', 378)

## Inspecting it

In [2]:
{
    "entries": uniarchive.entry_count("demo.zip"),
    "names": uniarchive.names("demo.zip"),
    "notes.txt": uniarchive.read_entry("demo.zip", "notes.txt"),
}

{'entries': 3,
 'names': ['notes.txt', 'data/', 'data/values.csv'],
 'notes.txt': b'one\n'}

## Extraction is transactional

Every payload is verified in a private staging tree; the destination appears
only once the whole archive has succeeded. Selectors pick exact files or whole
subtrees.

In [3]:
uniarchive.extract("demo.zip", "out")
# relpath returns the platform separator; the archive's own names use "/".
sorted(os.path.relpath(os.path.join(r, f), "out").replace(os.sep, "/")
       for r, _, fs in os.walk("out") for f in fs)

['data/values.csv', 'notes.txt']

In [4]:
uniarchive.extract("demo.zip", "partial", ["notes.txt"])
sorted(os.listdir("partial"))

['notes.txt']

## A refused output

In [5]:
try:
    uniarchive.create("demo.zip", ["notes.txt"])
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

ValueError: archive could not be created
